Install reqs

In [ ]:
!pip install -q meeko prody rdkit vina gemmi

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 294.0/294.0 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.5/37.5 MB 29.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.0/37.0 MB 14.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 70.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.1/103.1 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 59.2 MB/s eta 0:00:00


In [ ]:
!pip install py3Dmol pubchempy

In [ ]:
# Get our protein
!wget https://files.rcsb.org/download/6LU7.pdb

--2026-04-25 16:39:23--  https://files.rcsb.org/download/6LU7.pdb
Resolving files.rcsb.org (files.rcsb.org)... 18.239.18.95, 18.239.18.25, 18.239.18.50, ...
Connecting to files.rcsb.org (files.rcsb.org)|18.239.18.95|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: unspecified [text/plain]
Saving to: ‘6LU7.pdb’

6LU7.pdb                [  <=>               ] 233.51K   620KB/s    in 0.4s    

2026-04-25 16:39:24 (620 KB/s) - ‘6LU7.pdb’ saved [239112]



Prepare our protein

In [ ]:
from prody import parsePDB, writePDB

protein = parsePDB("6LU7.pdb")

# Keep protein chain A, remove waters and non-protein HETATM ligands
receptor = protein.select("chain A and protein")

writePDB("6LU7_receptor.pdb", receptor)

@> 2500 atoms and 1 coordinate set(s) were parsed in 0.02s.
DEBUG:.prody:2500 atoms and 1 coordinate set(s) were parsed in 0.02s.


'6LU7_receptor.pdb'

In [ ]:
!mk_prepare_receptor.py -i 6LU7_receptor.pdb -o 6LU7_receptor -p -v \
  --box_center -9.997 13.086 67.796 \
  --box_size 24 24 24

@> 2367 atoms and 1 coordinate set(s) were parsed in 0.02s.

Files written:
  6LU7_receptor.pdbqt <-- static (i.e., rigid) receptor input file
6LU7_receptor.box.txt <-- Vina-style box dimension file
6LU7_receptor.box.pdb <-- PDB file to visualize the grid box


Get the molecules we will screen

In [ ]:
from vina import Vina
from rdkit import Chem
from rdkit.Chem import AllChem, Descriptors, Crippen
import pandas as pd
import os

batch2_expansion_names = {
    # More Mpro / 3CLpro focused
    "N3": "N3 inhibitor",
    "X77": "X77",
    "ML188": "ML188",
    "ML300": "ML300",
    "11a": "SARS-CoV-2 Mpro inhibitor 11a",
    "11b": "SARS-CoV-2 Mpro inhibitor 11b",
    "13b": "SARS-CoV-2 Mpro inhibitor 13b",
    "MPI8": "MPI8 SARS-CoV-2 Mpro inhibitor",
    "MPI29": "MPI29 SARS-CoV-2 Mpro inhibitor",
    "NZ-804": "NZ-804",

    # HCV / viral protease inhibitors
    "Telaprevir": "Telaprevir",
    "Grazoprevir": "Grazoprevir",
    "Paritaprevir": "Paritaprevir",
    "Danoprevir": "Danoprevir",
    "Faldaprevir": "Faldaprevir",
    "Asunaprevir": "Asunaprevir",
    "Vaniprevir": "Vaniprevir",
    "Glecaprevir": "Glecaprevir",
    "Voxilaprevir": "Voxilaprevir",

    # Broad protease/cysteine protease inhibitors
    "Calpeptin": "Calpeptin",
    "MG-132": "MG-132",
    "ALLN": "ALLN",
    "E-64": "E-64",
    "E-64d": "E-64d",
    "Leupeptin": "Leupeptin",
    "Chymostatin": "Chymostatin",
    "Antipain": "Antipain",
    "Pepstatin A": "Pepstatin A",

    # Natural product comparators
    "Luteolin": "Luteolin",
    "Apigenin": "Apigenin",
    "Kaempferol": "Kaempferol",
    "Myricetin": "Myricetin",
    "Hesperetin": "Hesperetin",
    "Naringenin": "Naringenin",
    "Rutin": "Rutin",
    "Fisetin": "Fisetin",
    "Resveratrol": "Resveratrol",
    "Theaflavin": "Theaflavin",
    "Tannic acid": "Tannic acid",

    # Repurposing / negative-ish controls
    "Disulfiram": "Disulfiram",
    "Shikonin": "Shikonin",
    "Chlorpromazine": "Chlorpromazine",
    "Cinanserin": "Cinanserin",
    "Montelukast": "Montelukast",
    "Remdesivir": "Remdesivir",
    "Favipiravir": "Favipiravir",
    "Molnupiravir": "Molnupiravir",
    "Leritrelvir": "Leritrelvir",
    "Olgotrelvir": "Olgotrelvir",
    "Iscartrelvir": "Iscartrelvir",
    "Telaprevir": "Telaprevir",
    "Indinavir": "Indinavir",
    "Fosamprenavir": "Fosamprenavir",
    "Amprenavir": "Amprenavir",
    "Ribavirin": "Ribavirin",
    "Thiram": "Thiram",
    "Zinc pyrithione": "Zinc pyrithione",
    "Chloroquine": "Chloroquine",
    "Hydroxychloroquine": "Hydroxychloroquine",
    "Imatinib": "Imatinib",
    "Dasatinib": "Dasatinib",
    "Sunitinib": "Sunitinib",
    "Erlotinib": "Erlotinib",
    "Gefitinib": "Gefitinib",
    "Gallic acid": "Gallic acid",
    "Aniline": "Aniline",
    "Phenol": "Phenol",
    "Benzamide": "Benzamide",
    "Benzoic acid": "Benzoic acid",
    "Salicylic acid": "Salicylic acid",
    "Acetanilide": "Acetanilide",
    "Acetophenone": "Acetophenone",
    "Benzaldehyde": "Benzaldehyde",
    "Pyridine": "Pyridine",
    "Imidazole": "Imidazole",
    "Indole": "Indole",
    "Quinoline": "Quinoline",
    "Isoquinoline": "Isoquinoline",
    "Lidocaine": "Lidocaine",
    "Propranolol": "Propranolol",
    "Atenolol": "Atenolol",
    "Diphenhydramine": "Diphenhydramine",
    "Metformin": "Metformin",
    "Niclosamide": "Niclosamide",
    "Closantel": "Closantel",
    "Rafoxanide": "Rafoxanide",
    "Acetic acid": "Acetic acid",
    "Propionic acid": "Propionic acid",
    "Butyric acid": "Butyric acid",
    "Formamide": "Formamide",
    "Dimethylformamide": "Dimethylformamide"
}

In [ ]:
import pubchempy as pcp
def fetch_pubchem_smiles(query):
    try:
        compounds = pcp.get_compounds(query, "name")
        if not compounds:
            return None
        return compounds[0].canonical_smiles
    except Exception as e:
        print(f"Failed: {query} | {e}")
        return None


molecules= {}

for name, query in batch2_expansion_names.items():
    smi = fetch_pubchem_smiles(query)
    if smi is not None:
        molecules[name] = smi
    else:
        print("No SMILES found for:", name)

print(f"Fetched {len(molecules)} molecules")
molecules

No SMILES found for: N3


/tmp/ipykernel_468/1255338746.py:7: PubChemPyDeprecationWarning: canonical_smiles is deprecated: Use connectivity_smiles instead
  return compounds[0].canonical_smiles


No SMILES found for: 11a
No SMILES found for: 11b
No SMILES found for: 13b
No SMILES found for: MPI8
No SMILES found for: MPI29
Fetched 84 molecules


{'X77': 'CC(C)(C)C1=CC=C(C=C1)N(C(C2=CN=CC=C2)C(=O)NC3CCCCC3)C(=O)C4=CN=CN4',
 'ML188': 'CC(C)(C)C1=CC=C(C=C1)N(C(C2=CN=CC=C2)C(=O)NC(C)(C)C)C(=O)C3=CC=CO3',
 'ML300': 'C1CC1C(=O)NC2=CC=C(C=C2)N(CC3=CSC=C3)C(=O)CN4C5=CC=CC=C5N=N4',
 'NZ-804': 'C1CN(CCC1=C2C3=CC=CC=C3CS(=O)(=O)C4=CC=CC=C42)C(=O)C5=C6C(=CN=C5)C=CN6',
 'Telaprevir': 'CCCC(C(=O)C(=O)NC1CC1)NC(=O)C2C3CCCC3CN2C(=O)C(C(C)(C)C)NC(=O)C(C4CCCCC4)NC(=O)C5=NC=CN=C5',
 'Grazoprevir': 'CC(C)(C)C1C(=O)N2CC(CC2C(=O)NC3(CC3C=C)C(=O)NS(=O)(=O)C4CC4)OC5=NC6=C(C=CC(=C6)OC)N=C5CCCCCC7CC7OC(=O)N1',
 'Paritaprevir': 'CC1=CN=C(C=N1)C(=O)NC2CCCCCC=CC3CC3(NC(=O)C4CC(CN4C2=O)OC5=NC6=CC=CC=C6C7=CC=CC=C75)C(=O)NS(=O)(=O)C8CC8',
 'Danoprevir': 'CC(C)(C)OC(=O)NC1CCCCCC=CC2CC2(NC(=O)C3CC(CN3C1=O)OC(=O)N4CC5=C(C4)C(=CC=C5)F)C(=O)NS(=O)(=O)C6CC6',
 'Faldaprevir': 'CC(C)C(=O)NC1=NC(=CS1)C2=NC3=C(C=CC(=C3Br)OC)C(=C2)OC4CC(N(C4)C(=O)C(C(C)(C)C)NC(=O)OC5CCCC5)C(=O)NC6(CC6C=C)C(=O)O',
 'Asunaprevir': 'CC(C)(C)C(C(=O)N1CC(CC1C(=O)NC2(CC2C=C)C(=O)NS(=O)(=O)C3

Prepare the molecules for screening

In [ ]:
rows = []

for name, smiles in molecules.items():
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        print(f"Invalid SMILES for {name}")
        continue
    mol = Chem.AddHs(mol)
    params = AllChem.ETKDGv3()
    params.randomSeed = 42
    # params.maxAttempts = 1000
    params.useRandomCoords = True

    conf_id = AllChem.EmbedMolecule(mol, params)
    if (conf_id == -1):
      print(f"Skipping {name} due to issue embedding molecule")
    AllChem.MMFFOptimizeMolecule(mol)
    Chem.MolToMolFile(mol, f"{name}.sdf")
    command = f"mk_prepare_ligand.py -i {name}.sdf -o {name}.pdbqt"
    os.system(command)
    rows.append({
        "name": name,
        "MW": Descriptors.MolWt(mol),
        "LogP": Crippen.MolLogP(mol),
        "HBD": Descriptors.NumHDonors(mol),
        "HBA": Descriptors.NumHAcceptors(mol),
        "Rotatable Bonds": Descriptors.NumRotatableBonds(mol),
        "Heavy Atoms": mol.GetNumHeavyAtoms()
    })
    print(f"Prepared {name}")

Prepared X77
Prepared ML188
Prepared ML300
Prepared NZ-804
Prepared Telaprevir
Prepared Grazoprevir
Prepared Paritaprevir
Prepared Danoprevir
Prepared Faldaprevir
Prepared Asunaprevir
Prepared Vaniprevir
Prepared Glecaprevir
Prepared Voxilaprevir
Prepared Calpeptin
Prepared MG-132
Prepared ALLN
Prepared E-64
Prepared E-64d
Prepared Leupeptin
Prepared Chymostatin
Prepared Antipain
Prepared Pepstatin A
Prepared Luteolin
Prepared Apigenin
Prepared Kaempferol
Prepared Myricetin
Prepared Hesperetin
Prepared Naringenin
Prepared Rutin
Prepared Fisetin
Prepared Resveratrol
Prepared Theaflavin
Prepared Tannic acid
Prepared Disulfiram
Prepared Shikonin
Prepared Chlorpromazine
Prepared Cinanserin
Prepared Montelukast
Prepared Remdesivir
Prepared Favipiravir
Prepared Molnupiravir
Prepared Leritrelvir
Prepared Olgotrelvir
Prepared Iscartrelvir
Prepared Indinavir
Prepared Fosamprenavir
Prepared Amprenavir
Prepared Ribavirin
Prepared Thiram


[16:55:20] UFFTYPER: Unrecognized charge state for atom: 0
[16:55:20] UFFTYPER: Unrecognized atom type: Zn+2 (0)


Prepared Zinc pyrithione
Prepared Chloroquine
Prepared Hydroxychloroquine
Prepared Imatinib
Prepared Dasatinib
Prepared Sunitinib
Prepared Erlotinib
Prepared Gefitinib
Prepared Gallic acid
Prepared Aniline
Prepared Phenol
Prepared Benzamide
Prepared Benzoic acid
Prepared Salicylic acid
Prepared Acetanilide
Prepared Acetophenone
Prepared Benzaldehyde
Prepared Pyridine
Prepared Imidazole
Prepared Indole
Prepared Quinoline
Prepared Isoquinoline
Prepared Lidocaine
Prepared Propranolol
Prepared Atenolol
Prepared Diphenhydramine
Prepared Metformin
Prepared Niclosamide
Prepared Closantel
Prepared Rafoxanide
Prepared Acetic acid
Prepared Propionic acid
Prepared Butyric acid
Prepared Formamide
Prepared Dimethylformamide


Do a light initial pass for screening

In [ ]:
from concurrent.futures import ProcessPoolExecutor, as_completed

RECEPTOR = "6LU7_receptor.pdbqt"
CENTER = [-10.7, 12.4, 68.8]
BOX = [22, 22, 22]

def dock_one(name, exhaustiveness=2, n_poses=1):
    try:
        print(f"Docking {name}...")

        v = Vina(sf_name="vina")
        v.set_receptor(RECEPTOR)
        v.set_ligand_from_file(f"{name}.pdbqt")
        v.compute_vina_maps(center=CENTER, box_size=BOX)

        v.dock(exhaustiveness=exhaustiveness, n_poses=n_poses)
        score = v.energies(n_poses=n_poses)[0][0]

        v.write_poses(f"{name}_docked.pdbqt", n_poses=n_poses, overwrite=True)

        return {
            "molecule": name,
            "vina_score_kcal_mol": score,
            # "ligand_efficiency": score/
            "status": "success"
        }

    except Exception as e:
        return {
            "molecule": name,
            "vina_score_kcal_mol": None,
            # "ligand_efficiency": None,
            "status": f"failed: {e}"
        }

print("Started initial screening...")

results = []

with ProcessPoolExecutor(max_workers=2) as executor:
    futures = [executor.submit(dock_one, name) for name in molecules]

    for future in as_completed(futures):
        result = future.result()
        results.append(result)
        print(result["molecule"], result["status"])

df = pd.DataFrame(results).sort_values("vina_score_kcal_mol").merge(pd.DataFrame(rows), left_on="molecule", right_on="name")
df

Started initial screening...
Docking X77...
Docking ML188...
Docking ML300...
ML188 success
Docking NZ-804...
X77 success
Docking Telaprevir...
ML300 success
Docking Grazoprevir...
NZ-804 success
Docking Paritaprevir...
Telaprevir success
Docking Danoprevir...
Grazoprevir success
Docking Faldaprevir...
Paritaprevir success
Docking Asunaprevir...
Danoprevir success
Docking Vaniprevir...
Faldaprevir success
Docking Glecaprevir...
Asunaprevir success
Docking Voxilaprevir...
Vaniprevir success
Docking Calpeptin...
Glecaprevir success
Docking MG-132...
Calpeptin success
Docking ALLN...
MG-132 success
Docking E-64...
ALLN success
Docking E-64d...
E-64 success
Docking Leupeptin...
E-64d success
Docking Chymostatin...
Leupeptin success
Docking Antipain...
Chymostatin success
Docking Pepstatin A...
Docking Luteolin...
Voxilaprevir success
Pepstatin A failed: Error: file Pepstatin A.pdbqt does not exist.
Docking Apigenin...
Luteolin success
Docking Kaempferol...
Apigenin success
Docking Myriceti

,molecule,vina_score_kcal_mol,status,name,MW,LogP,HBD,HBA,Rotatable Bonds,Heavy Atoms
0,Rutin,-9.158,success,Rutin,610.521,-1.87880,10,16,16,43
1,Theaflavin,-8.844,success,Theaflavin,564.499,2.21340,9,12,11,41
2,Imatinib,-8.716,success,Imatinib,493.615,4.59032,2,7,7,37
3,NZ-804,-8.473,success,NZ-804,469.566,4.58830,1,4,1,34
4,Leritrelvir,-8.347,success,Leritrelvir,639.716,1.87980,4,6,10,45
...,...,...,...,...,...,...,...,...,...,...
79,Benzoic acid,NaN,failed: Error: file Benzoic acid.pdbqt does no...,Benzoic acid,122.123,1.38480,1,2,1,9
80,Salicylic acid,NaN,failed: Error: file Salicylic acid.pdbqt does ...,Salicylic acid,138.122,1.09040,2,3,2,10
81,Acetic acid,NaN,failed: Error: file Acetic acid.pdbqt does not...,Acetic acid,60.052,0.09090,1,2,0,4
82,Propionic acid,NaN,failed: Error: file Propionic acid.pdbqt does ...,Propionic acid,74.079,0.48100,1,2,1,5


Find our strongest matches

In [ ]:
df["ligand_efficiency"] = df["vina_score_kcal_mol"] / df["Heavy Atoms"]
df = df.sort_values(["vina_score_kcal_mol", "ligand_efficiency"])
df

,molecule,vina_score_kcal_mol,status,name,MW,LogP,HBD,HBA,Rotatable Bonds,Heavy Atoms,ligand_efficiency
0,Rutin,-9.158,success,Rutin,610.521,-1.87880,10,16,16,43,-0.212977
1,Theaflavin,-8.844,success,Theaflavin,564.499,2.21340,9,12,11,41,-0.215707
2,Imatinib,-8.716,success,Imatinib,493.615,4.59032,2,7,7,37,-0.235568
3,NZ-804,-8.473,success,NZ-804,469.566,4.58830,1,4,1,34,-0.249206
4,Leritrelvir,-8.347,success,Leritrelvir,639.716,1.87980,4,6,10,45,-0.185489
...,...,...,...,...,...,...,...,...,...,...,...
79,Benzoic acid,NaN,failed: Error: file Benzoic acid.pdbqt does no...,Benzoic acid,122.123,1.38480,1,2,1,9,NaN
80,Salicylic acid,NaN,failed: Error: file Salicylic acid.pdbqt does ...,Salicylic acid,138.122,1.09040,2,3,2,10,NaN
81,Acetic acid,NaN,failed: Error: file Acetic acid.pdbqt does not...,Acetic acid,60.052,0.09090,1,2,0,4,NaN
82,Propionic acid,NaN,failed: Error: file Propionic acid.pdbqt does ...,Propionic acid,74.079,0.48100,1,2,1,5,NaN


In [ ]:
strong_candidates = df[(df["vina_score_kcal_mol"] <= -7.5) & (df["ligand_efficiency"] <= -0.2)]

In [ ]:
strong_candidates

,molecule,vina_score_kcal_mol,status,name,MW,LogP,HBD,HBA,Rotatable Bonds,Heavy Atoms,ligand_efficiency
0,Rutin,-9.158,success,Rutin,610.521,-1.87880,10,16,16,43,-0.212977
1,Theaflavin,-8.844,success,Theaflavin,564.499,2.21340,9,12,11,41,-0.215707
2,Imatinib,-8.716,success,Imatinib,493.615,4.59032,2,7,7,37,-0.235568
3,NZ-804,-8.473,success,NZ-804,469.566,4.58830,1,4,1,34,-0.249206
6,X77,-8.230,success,X77,459.594,4.93920,2,4,6,34,-0.242059
8,Olgotrelvir,-7.897,success,Olgotrelvir,494.570,0.52960,6,7,12,34,-0.232265
9,Iscartrelvir,-7.854,success,Iscartrelvir,526.391,4.41820,3,6,6,34,-0.231000
10,Kaempferol,-7.819,success,Kaempferol,286.239,2.30530,4,6,5,21,-0.372333
11,Apigenin,-7.757,success,Apigenin,270.240,2.41960,3,5,4,20,-0.387850
12,ML300,-7.744,success,ML300,431.521,4.07470,1,5,7,31,-0.249806


Perform a more stringent second round

In [ ]:
results = []

with ProcessPoolExecutor(max_workers=2) as executor:
    futures = [executor.submit(dock_one, name, exhaustiveness=16, n_poses=10) for name in strong_candidates["molecule"]]

    for future in as_completed(futures):
        result = future.result()
        results.append(result)
        print(result["molecule"], result["status"])

df_stage2 = pd.DataFrame(results).sort_values("vina_score_kcal_mol").merge(pd.DataFrame(rows), left_on="molecule", right_on="name")
df_stage2

Docking Rutin...
Docking Theaflavin...
Docking Imatinib...
Theaflavin success
Docking NZ-804...
Imatinib success
Docking X77...
Rutin success
Docking Olgotrelvir...
NZ-804 success
Docking Iscartrelvir...
X77 success
Docking Kaempferol...
Olgotrelvir success
Docking Apigenin...
Kaempferol success
Docking ML300...
Apigenin success
Docking Gefitinib...
Iscartrelvir success
Docking Fisetin...
ML300 success
Gefitinib success
Fisetin success


,molecule,vina_score_kcal_mol,status,name,MW,LogP,HBD,HBA,Rotatable Bonds,Heavy Atoms
0,Rutin,-9.146,success,Rutin,610.521,-1.87880,10,16,16,43
1,Imatinib,-8.866,success,Imatinib,493.615,4.59032,2,7,7,37
2,Theaflavin,-8.856,success,Theaflavin,564.499,2.21340,9,12,11,41
3,NZ-804,-8.511,success,NZ-804,469.566,4.58830,1,4,1,34
4,Olgotrelvir,-7.998,success,Olgotrelvir,494.570,0.52960,6,7,12,34
5,Iscartrelvir,-7.876,success,Iscartrelvir,526.391,4.41820,3,6,6,34
6,Kaempferol,-7.834,success,Kaempferol,286.239,2.30530,4,6,5,21
7,Apigenin,-7.769,success,Apigenin,270.240,2.41960,3,5,4,20
8,ML300,-7.695,success,ML300,431.521,4.07470,1,5,7,31
9,X77,-7.646,success,X77,459.594,4.93920,2,4,6,34


In [ ]:
df_stage2["ligand_efficiency"] = df_stage2["vina_score_kcal_mol"] / df_stage2["Heavy Atoms"]
df_stage2 = df_stage2.sort_values(["vina_score_kcal_mol", "ligand_efficiency"])
strong_candidates_2 = df_stage2[
    (df_stage2["vina_score_kcal_mol"] <= -7.8) &
    (df_stage2["ligand_efficiency"] <= -0.23) &
    (df_stage2["MW"] <= 550) &
    (df_stage2["Rotatable Bonds"] <= 12) &
    (df_stage2["Heavy Atoms"] <= 40) &
    (df_stage2["HBD"] <= 6) &
    (df_stage2["HBA"] <= 10) &
    (df_stage2["LogP"] <= 5.0)
].copy()

In [ ]:
strong_candidates_2

,molecule,vina_score_kcal_mol,status,name,MW,LogP,HBD,HBA,Rotatable Bonds,Heavy Atoms,ligand_efficiency
1,Imatinib,-8.866,success,Imatinib,493.615,4.59032,2,7,7,37,-0.239622
3,NZ-804,-8.511,success,NZ-804,469.566,4.58830,1,4,1,34,-0.250324
4,Olgotrelvir,-7.998,success,Olgotrelvir,494.570,0.52960,6,7,12,34,-0.235235
5,Iscartrelvir,-7.876,success,Iscartrelvir,526.391,4.41820,3,6,6,34,-0.231647
6,Kaempferol,-7.834,success,Kaempferol,286.239,2.30530,4,6,5,21,-0.373048
